# PHASE 2: EXPLORATORY DATA ANALYSIS (2D PNG SLICES)

**Objective:** Analyze liver CT slices and tumor masks to understand data characteristics, measure class imbalance, and inform modeling decisions.

**Hardware:** NVIDIA RTX 3050 Ti 4GB
**Depends on:** Phase 1 Data Loading (split files)
**Target Size:** 256x256 (4x VRAM reduction vs 512x512)

---

## Table of Contents
1. GPU Setup & Imports
2. Load Data & Splits
3. Image Intensity Statistics
4. Tumor Coverage Analysis
5. Class Imbalance (Per-Split)
6. Volume-Level Statistics
7. Sample Visualizations
8. Morphology Analysis
9. Save Statistics JSON
10. Key Insights & Recommendations

## [SETUP] Cell 1: GPU Setup & Imports

In [ ]:
import os, sys, json, random, time
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from skimage import measure

plt.style.use('seaborn-v0_8-whitegrid')

# Ensure src/ is importable (walk up from cwd until src/ found)
project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import from src/
from src.data_loader import DatasetConfig, DataPathManager
from src.gpu_utils import DEVICE, gpu_report, gpu_clear, to_tensor, to_numpy, batch_to_tensor

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# GPU Info
print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA: {torch.version.cuda}")
    torch.backends.cudnn.benchmark = True
else:
    print("[WARNING] Using CPU (slower)")
print("=" * 60)
gpu_report()

# Create EDA output directories
EDA_OUTPUT_DIR = DatasetConfig.EDA_OUTPUT_DIR
EDA_PLOTS_DIR = DatasetConfig.EDA_PLOTS_DIR
EDA_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n[INFO] EDA outputs: {EDA_OUTPUT_DIR}")
print(f"[INFO] EDA plots: {EDA_PLOTS_DIR}")

## [DATA] Cell 2: Load Data & Splits

In [ ]:
# Build volume index from PNG directories
path_manager = DataPathManager()
volume_index = path_manager.build_index()

total_slices = sum(len(v) for v in volume_index['image_paths'].values())
print(f"[INFO] Volume Index: {len(volume_index['volumes'])} volumes, {total_slices:,} slices")

# Load splits from Phase 1
def load_split_vids(split_name: str) -> List[int]:
    path = DatasetConfig.SPLITS_DIR / f"{split_name}_volumes.txt"
    with open(path, 'r') as f:
        return [int(x) for x in f.read().strip().split(',') if x]

splits = {name: load_split_vids(name) for name in ['train', 'val', 'test']}

print("\n[INFO] Split Summary:")
for name, vids in splits.items():
    n_slices = sum(len(volume_index['image_paths'].get(v, [])) for v in vids)
    print(f"   {name}: {len(vids):>4} volumes, {n_slices:>6,} slices")

train_vids, val_vids, test_vids = splits['train'], splits['val'], splits['test']

## [INTENSITY] Cell 3: Image Intensity Statistics (GPU-Accelerated)

In [ ]:
def analyze_intensity(volume_index, vids, n_samples=2000):
    """GPU-accelerated intensity statistics."""
    all_pairs = [(vid, sid) for vid in vids if vid in volume_index['image_paths']
                 for sid in range(len(volume_index['image_paths'][vid]))]
    sampled = random.sample(all_pairs, min(n_samples, len(all_pairs)))

    imgs = []
    for vid, sid in sampled:
        path = volume_index['image_paths'][vid][sid]
        img = np.array(Image.open(path).convert('L'), dtype=np.float32) / 255.0
        imgs.append(img)

    tensor = batch_to_tensor(imgs)  # GPU: (B, H, W)
    flat = tensor.flatten()

    stats = {
        'mean': float(to_numpy(flat.mean())),
        'std': float(to_numpy(flat.std())),
        'min': float(to_numpy(flat.min())),
        'max': float(to_numpy(flat.max())),
        'samples': len(sampled),
    }

        # Flatten images before deleting them
    all_intensities = np.concatenate([img.flatten() for img in imgs])
    
    del tensor, flat, imgs
    gpu_clear()
    
    return stats, all_intensities

print("[STAT] Computing intensity statistics...")
intensity_stats, all_intensities = analyze_intensity(volume_index, train_vids)

print(f"   Samples: {intensity_stats['samples']:,}")
print(f"   Mean: {intensity_stats['mean']:.4f}")
print(f"   Std:  {intensity_stats['std']:.4f}")
print(f"   Min:  {intensity_stats['min']:.4f}")
print(f"   Max:  {intensity_stats['max']:.4f}")

# Plot: Histogram + Boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(all_intensities, bins=80, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(intensity_stats['mean'], color='red', linestyle='--', linewidth=2,
                label=f"Mean: {intensity_stats['mean']:.3f}")
axes[0].set_xlabel("Normalized Intensity (0-1)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Image Intensity Distribution")
axes[0].legend()

axes[1].boxplot(all_intensities, vert=True, tick_labels=['All Slices'])
axes[1].set_ylabel("Pixel Intensity (0-1)")
axes[1].set_title("Intensity Box Plot")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(EDA_PLOTS_DIR / '01_intensity_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {EDA_PLOTS_DIR / '01_intensity_histogram.png'}")

## [TUMOR] Cell 4: Tumor Coverage Analysis (GPU-Accelerated)

In [ ]:
def analyze_tumor_coverage(volume_index, vids, n_samples=5000):
    """GPU-accelerated tumor pixel analysis. target_size=(256,256) keeps VRAM ~1.3GB."""
    all_pairs = [(vid, sid) for vid in vids if vid in volume_index['mask_paths']
                 for sid in range(len(volume_index['mask_paths'][vid]))]
    sampled = random.sample(all_pairs, min(n_samples, len(all_pairs)))

    masks = []
    for vid, sid in sampled:
        path = volume_index['mask_paths'][vid][sid]
        mask = np.array(Image.open(path).convert('L'), dtype=np.float32) / 255.0
        masks.append((mask > 0).astype(np.float32))

    tensor = batch_to_tensor(masks)  # GPU: (B, H, W)
    tumor_px = to_numpy(tensor.sum(dim=(1, 2)))  # per-slice tumor pixels
    total_px = tensor[0].numel()

    has_tumor = tumor_px > 0
    coverage = tumor_px / total_px * 100

    stats = {
        'total_samples': len(sampled),
        'slices_with_tumor': int(has_tumor.sum()),
        'slices_without_tumor': int((~has_tumor).sum()),
        'tumor_slice_pct': float(has_tumor.sum() / len(has_tumor) * 100),
        'mean_coverage_pct': float(coverage.mean()),
        'median_coverage_pct': float(np.median(coverage)),
        'max_coverage_pct': float(coverage.max()),
        'std_coverage_pct': float(coverage.std()),
        'mean_tumor_pixels': float(tumor_px.mean()),
        'max_tumor_pixels': int(tumor_px.max()),
    }

    del tensor, masks
    gpu_clear()
    return stats, coverage

print("[STAT] Computing tumor coverage...")
tumor_stats, tumor_coverage = analyze_tumor_coverage(volume_index, train_vids)

print(f"   Slices with tumor:    {tumor_stats['slices_with_tumor']:>5,} ({tumor_stats['tumor_slice_pct']:.1f}%)")
print(f"   Slices without tumor: {tumor_stats['slices_without_tumor']:>5,} ({100-tumor_stats['tumor_slice_pct']:.1f}%)")
print(f"   Mean coverage: {tumor_stats['mean_coverage_pct']:.3f}%")
print(f"   Max coverage:  {tumor_stats['max_coverage_pct']:.3f}%")

# Plot: Coverage distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(tumor_coverage, bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[0].axvline(tumor_stats['mean_coverage_pct'], color='red', linestyle='--',
                label=f"Mean: {tumor_stats['mean_coverage_pct']:.3f}%")
axes[0].set_xlabel("Tumor Coverage (%)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Tumor Coverage (All Slices)")
axes[0].legend()

tumor_only = tumor_coverage[tumor_coverage > 0]
if len(tumor_only) > 0:
    axes[1].hist(tumor_only, bins=30, color='forestgreen', edgecolor='black', alpha=0.7)
    axes[1].axvline(tumor_only.mean(), color='red', linestyle='--',
                    label=f"Mean: {tumor_only.mean():.3f}%")
    axes[1].set_xlabel("Tumor Coverage (%)")
    axes[1].set_ylabel("Frequency")
    axes[1].set_title(f"Tumor Coverage (WITH Tumor, n={len(tumor_only)})")
    axes[1].legend()

axes[2].boxplot([tumor_coverage, tumor_only] if len(tumor_only) > 0 else [tumor_coverage],
                tick_labels=['All Slices', 'Tumor Only'][:2 if len(tumor_only) > 0 else 1])
axes[2].set_ylabel("Coverage (%)")
axes[2].set_title("Tumor Coverage Box Plot")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(EDA_PLOTS_DIR / '02_tumor_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {EDA_PLOTS_DIR / '02_tumor_coverage.png'}")

## [IMBALANCE] Cell 5: Class Imbalance Analysis (Per-Split Comparison)

In [ ]:
def analyze_class_imbalance(volume_index, vids, n_samples=2000):
    """GPU-accelerated background-to-tumor ratio."""
    all_pairs = [(vid, sid) for vid in vids if vid in volume_index['mask_paths']
                 for sid in range(len(volume_index['mask_paths'][vid]))]
    sampled = random.sample(all_pairs, min(n_samples, len(all_pairs)))

    masks = []
    for vid, sid in sampled:
        path = volume_index['mask_paths'][vid][sid]
        mask = np.array(Image.open(path).convert('L'), dtype=np.float32) / 255.0
        masks.append((mask > 0).astype(np.float32))

    tensor = batch_to_tensor(masks)
    total_px = tensor.numel()
    tumor_px = float(to_numpy(tensor.sum()))
    bg_px = total_px - tumor_px

    stats = {
        'samples': len(sampled),
        'total_pixels': total_px,
        'background_pct': bg_px / total_px * 100,
        'tumor_pct': tumor_px / total_px * 100,
        'imbalance_ratio': bg_px / max(tumor_px, 1),
    }

    del tensor, masks
    gpu_clear()
    return stats

print("[STAT] Computing per-split class imbalance...")
imbalance_by_split = {}
for name in ['train', 'val', 'test']:
    stats = analyze_class_imbalance(volume_index, splits[name])
    imbalance_by_split[name] = stats
    print(f"   {name}: Tumor={stats['tumor_pct']:.4f}%  |  "
          f"BG={stats['background_pct']:.2f}%  |  "
          f"Ratio=1:{stats['imbalance_ratio']:.0f}")

# Plot: Pie chart (train) + Grouped bar (all splits)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = ['Background', 'Tumor']
colors = ['#3498db', '#e74c3c']
tr = imbalance_by_split['train']
axes[0].pie([tr['background_pct'], tr['tumor_pct']], explode=(0, 0.1),
            labels=labels, colors=colors, autopct='%1.3f%%',
            shadow=True, startangle=90)
axes[0].set_title('Class Distribution (Train Split, Pixel-Level)')

split_names = list(imbalance_by_split.keys())
bg_vals = [imbalance_by_split[s]['background_pct'] for s in split_names]
tu_vals = [imbalance_by_split[s]['tumor_pct'] for s in split_names]
x = np.arange(len(split_names))
w = 0.35
axes[1].bar(x - w/2, bg_vals, w, label='Background', color='#3498db', edgecolor='black')
axes[1].bar(x + w/2, tu_vals, w, label='Tumor', color='#e74c3c', edgecolor='black')
for i, (b, t) in enumerate(zip(bg_vals, tu_vals)):
    axes[1].text(i - w/2, b, f'{b:.1f}%', ha='center', va='bottom', fontsize=9)
    axes[1].text(i + w/2, t, f'{t:.3f}%', ha='center', va='bottom', fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(split_names)
axes[1].set_ylabel("Pixel Percentage")
axes[1].set_title("Class Imbalance Across Splits")
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(EDA_PLOTS_DIR / '03_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {EDA_PLOTS_DIR / '03_class_imbalance.png'}")

## [VOLUME] Cell 6: Volume-Level Statistics

In [ ]:
def analyze_volume_stats(volume_index, vids, max_slices=400):
    """Compute per-volume statistics with sampling for VRAM safety."""
    results = {'slices_per_volume': [], 'tumor_pixels': [], 'slices_with_tumor': []}

    for vid in vids:
        if vid not in volume_index['image_paths']:
            continue
        n = len(volume_index['image_paths'][vid])
        results['slices_per_volume'].append(n)

        if vid in volume_index['mask_paths']:
            step = max(1, n // max_slices)
            sidxs = list(range(0, n, step))[:max_slices]
            masks = []
            for sid in sidxs:
                m = np.array(Image.open(volume_index['mask_paths'][vid][sid]).convert('L'),
                             dtype=np.float32) / 255.0
                masks.append((m > 0).astype(np.float32))
            if masks:
                tensor = batch_to_tensor(masks)
                px_per_slice = to_numpy(tensor.sum(dim=(1, 2)))
                results['tumor_pixels'].append(int(px_per_slice.sum()))
                results['slices_with_tumor'].append(int((px_per_slice > 0).sum()))
                del tensor, masks
                gpu_clear()

    return results

print("[STAT] Computing volume-level statistics...")
vol_stats = {name: analyze_volume_stats(volume_index, splits[name])
             for name in ['train', 'val', 'test']}

for name in ['train', 'val', 'test']:
    s = vol_stats[name]
    spv = np.array(s['slices_per_volume'])
    print(f"   {name}: n={len(spv)} volumes, slices/vol=[{spv.min():.0f}-{spv.max():.0f}], "
          f"mean={spv.mean():.1f}")

# Plot: 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors_map = {'train': '#2ecc71', 'val': '#3498db', 'test': '#e74c3c'}

# Top-left: Slices per volume histogram
for name in ['train', 'val', 'test']:
    axes[0, 0].hist(vol_stats[name]['slices_per_volume'], bins=20,
                    color=colors_map[name], alpha=0.6, label=name)
axes[0, 0].set_xlabel('Slices per Volume')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Slices per Volume Distribution')
axes[0, 0].legend()

# Top-right: Tumor pixels boxplot
data = [vol_stats[name]['tumor_pixels'] for name in ['train', 'val', 'test']]
axes[0, 1].boxplot(data, tick_labels=['Train', 'Val', 'Test'])
axes[0, 1].set_ylabel('Total Tumor Pixels')
axes[0, 1].set_title('Tumor Pixels per Volume')
axes[0, 1].grid(True, alpha=0.3)

# Bottom-left: Tumor ratio per split
ratios = []
for name in ['train', 'val', 'test']:
    sv = np.array(vol_stats[name]['slices_with_tumor'])
    spv = np.array(vol_stats[name]['slices_per_volume'])
    ratios.append(np.mean(sv / spv * 100))
bars = axes[1, 0].bar(['Train', 'Val', 'Test'], ratios,
                      color=[colors_map[n] for n in ['train', 'val', 'test']],
                      edgecolor='black')
for bar, r in zip(bars, ratios):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{r:.1f}%', ha='center', va='bottom')
axes[1, 0].set_ylabel('Mean Slices with Tumor (%)')
axes[1, 0].set_title('Tumor Presence per Volume')

# Bottom-right: Scatter plot
for name in ['train', 'val', 'test']:
    axes[1, 1].scatter(vol_stats[name]['slices_per_volume'],
                       vol_stats[name]['tumor_pixels'],
                       alpha=0.5, c=colors_map[name], label=name)
axes[1, 1].set_xlabel('Slices per Volume')
axes[1, 1].set_ylabel('Total Tumor Pixels')
axes[1, 1].set_title('Volume Size vs Tumor Load')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(EDA_PLOTS_DIR / '04_volume_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {EDA_PLOTS_DIR / '04_volume_analysis.png'}")

## [VISUAL] Cell 7: Sample Visualizations

In [ ]:
def show_random_samples(volume_index, vids, n_samples=6):
    """Display random slices with tumor overlay."""
    all_pairs = [(vid, sid) for vid in vids if vid in volume_index['mask_paths']
                 for sid in range(len(volume_index['mask_paths'][vid]))]
    sampled = random.sample(all_pairs, min(n_samples, len(all_pairs)))

    cols, rows = 3, (n_samples + 2) // 3
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    axes = axes.flatten()

    for i, (vid, sid) in enumerate(sampled):
        img = np.array(Image.open(volume_index['image_paths'][vid][sid]).convert('L'))
        mask = np.array(Image.open(volume_index['mask_paths'][vid][sid]).convert('L'))
        mask_bin = (mask > 0).astype(float)
        overlay = np.ma.masked_where(mask_bin == 0, mask_bin)

        axes[i].imshow(img, cmap='gray')
        axes[i].imshow(overlay, cmap='Reds', alpha=0.5)
        tumor_pct = mask_bin.sum() / mask_bin.size * 100
        tag = "WITH tumor" if tumor_pct > 0 else "NO tumor"
        axes[i].set_title(f"Vol {vid:03d}, Slice {sid:03d}\nTumor: {tumor_pct:.2f}% ({tag})")
        axes[i].axis('off')

    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR / '05_random_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[OK] Saved: {EDA_PLOTS_DIR / '05_random_samples.png'}")

def show_volume_sequence(volume_index, vid, n_slices=8):
    """Display evenly spaced slices from one volume."""
    if vid not in volume_index['image_paths']:
        print(f"Volume {vid} not found")
        return
    total = len(volume_index['image_paths'][vid])
    indices = np.linspace(0, total - 1, n_slices, dtype=int)

    cols = 4
    rows = (n_slices + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4*rows))
    axes = axes.flatten()

    for i, sid in enumerate(indices):
        img = np.array(Image.open(volume_index['image_paths'][vid][sid]).convert('L'))
        axes[i].imshow(img, cmap='gray')
        if vid in volume_index['mask_paths'] and sid < len(volume_index['mask_paths'][vid]):
            mask = np.array(Image.open(volume_index['mask_paths'][vid][sid]).convert('L'))
            mask_bin = (mask > 0).astype(float)
            overlay = np.ma.masked_where(mask_bin == 0, mask_bin)
            axes[i].imshow(overlay, cmap='Reds', alpha=0.5)
            pct = mask_bin.sum() / mask_bin.size * 100
        else:
            pct = 0.0
        axes[i].set_title(f"Slice {sid:03d} | Tumor: {pct:.2f}%")
        axes[i].axis('off')

    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    plt.suptitle(f"Volume {vid} — {total} total slices", fontsize=14)
    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR / '06_volume_sequence.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[OK] Saved: {EDA_PLOTS_DIR / '06_volume_sequence.png'}")

print("[VISUAL] Random samples WITH tumor:")
tumor_vids = [v for v in train_vids if v in volume_index['mask_paths']]
show_random_samples(volume_index, tumor_vids)

print("\n[VISUAL] Volume sequence WITH tumor (Volume 5):")
show_volume_sequence(volume_index, vid=5)

print("\n[VISUAL] Volume sequence WITHOUT tumor (Volume 10):")
show_volume_sequence(volume_index, vid=10)

## [MORPHOLOGY] Cell 8: Connected Components Analysis

In [ ]:
def analyze_morphology(volume_index, vids, n_samples=2000):
    """Analyze tumor region morphology: component count, size, and aspect ratio."""
    all_pairs = [(vid, sid) for vid in vids if vid in volume_index['mask_paths']
                 for sid in range(len(volume_index['mask_paths'][vid]))]
    sampled = random.sample(all_pairs, min(n_samples, len(all_pairs)))

    n_components = []
    region_sizes = []
    aspect_ratios = []

    for vid, sid in sampled:
        path = volume_index['mask_paths'][vid][sid]
        mask = np.array(Image.open(path).convert('L'), dtype=np.float32) / 255.0
        mask_bin = (mask > 0).astype(np.uint8)

        # GPU pre-filter: skip empty slices
        if torch.cuda.is_available():
            if to_tensor(mask_bin).sum().item() == 0:
                continue
        elif mask_bin.sum() == 0:
            continue

        labeled = measure.label(mask_bin)
        n_components.append(labeled.max())

        props = measure.regionprops(labeled)
        for p in props:
            region_sizes.append(p.area)
            # bbox: (min_row, min_col, max_row, max_col)
            h = p.bbox[2] - p.bbox[0]
            w = p.bbox[3] - p.bbox[1]
            aspect_ratios.append(max(w, h) / max(min(w, h), 1))

    gpu_clear()

    stats = {
        'slices_analyzed': len(n_components),
        'mean_components': float(np.mean(n_components)) if n_components else 0,
        'max_components': int(np.max(n_components)) if n_components else 0,
        'mean_region_size': float(np.mean(region_sizes)) if region_sizes else 0,
        'max_region_size': int(np.max(region_sizes)) if region_sizes else 0,
        'mean_aspect_ratio': float(np.mean(aspect_ratios)) if aspect_ratios else 0,
    }
    return stats, region_sizes, aspect_ratios

print("[STAT] Computing morphology statistics...")
morpho_stats, region_sizes, aspect_ratios = analyze_morphology(volume_index, train_vids)

print(f"   Slices with tumor: {morpho_stats['slices_analyzed']}")
print(f"   Mean components/slice: {morpho_stats['mean_components']:.2f}")
print(f"   Mean region size: {morpho_stats['mean_region_size']:.0f} px")
print(f"   Mean aspect ratio: {morpho_stats['mean_aspect_ratio']:.2f}")

# Plot: Region size + Aspect ratio
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if region_sizes:
    axes[0].hist(region_sizes, bins=50, color='purple', edgecolor='black', alpha=0.7)
    axes[0].axvline(np.mean(region_sizes), color='red', linestyle='--',
                    label=f"Mean: {np.mean(region_sizes):.0f}")
    axes[0].set_xlabel("Region Size (pixels)")
    axes[0].set_ylabel("Frequency")
    axes[0].set_title("Tumor Region Size Distribution")
    axes[0].legend()

if aspect_ratios:
    axes[1].scatter(region_sizes, aspect_ratios, alpha=0.3, c='darkorange', edgecolors='none')
    axes[1].axhline(1.0, color='gray', linestyle='--', alpha=0.5, label="Perfect circle")
    axes[1].set_xlabel("Region Size (pixels)")
    axes[1].set_ylabel("Aspect Ratio (W/H)")
    axes[1].set_title("Region Size vs Aspect Ratio")
    axes[1].set_xscale('log')
    axes[1].legend()

plt.tight_layout()
plt.savefig(EDA_PLOTS_DIR / '07_connected_components.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {EDA_PLOTS_DIR / '07_connected_components.png'}")

## [2B] Enhanced Volume Metadata Analysis

**Purpose:** Per-split breakdown, volume summary statistics, and CSV export

In [ ]:
import pandas as pd

def build_volume_dataframe(volume_index, splits):
    rows = []
    for split_name, vol_ids in splits.items():
        for vol_id in vol_ids:
            if vol_id not in volume_index['image_paths']:
                continue
            num_slices = len(volume_index['image_paths'][vol_id])
            mask_exists = vol_id in volume_index['mask_paths']
            rows.append({
                'volume_id': vol_id,
                'split': split_name,
                'total_slices': num_slices,
                'has_masks': mask_exists,
            })
    df = pd.DataFrame(rows)
    return df

volume_df = build_volume_dataframe(volume_index, splits)
print(f"[DATA] Volume metadata: {len(volume_df)} volumes across 3 splits")
print(volume_df.groupby('split').agg(
    count=('volume_id', 'count'),
    avg_slices=('total_slices', 'mean'),
    min_slices=('total_slices', 'min'),
    max_slices=('total_slices', 'max')
).to_string())

# Bar chart: average slices per volume by split
fig, ax = plt.subplots(figsize=(8, 5))
grouped = volume_df.groupby('split')['total_slices'].mean()
colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax.bar(grouped.index, grouped.values, color=colors, edgecolor='black', width=0.5)
for bar, val in zip(bars, grouped.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.1f}', ha='center', fontsize=11)
ax.set_ylabel('Average Slices per Volume')
ax.set_title('Average Volume Size by Split')
plt.tight_layout()
plt.savefig(EDA_PLOTS_DIR / '08_split_volume_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {EDA_PLOTS_DIR / '08_split_volume_sizes.png'}")

# Save volume_df to CSV for downstream phases
csv_path = EDA_OUTPUT_DIR / 'volume_metadata.csv'
volume_df.to_csv(csv_path, index=False)
print(f"[OK] Saved: {csv_path}")

# Per-split enhanced stats
per_split = {}
for name in ['train', 'val', 'test']:
    sub = volume_df[volume_df['split'] == name]
    per_split[name] = {
        'count': int(len(sub)),
        'avg_slices': float(sub['total_slices'].mean()),
        'min_slices': int(sub['total_slices'].min()),
        'max_slices': int(sub['total_slices'].max()),
    }

enhanced_stats = {
    'total_volumes': int(len(volume_df)),
    'per_split': per_split,
}

## [3B] HU Windowing Analysis
**Purpose:** Determine intensity percentiles and compare HU window options


In [ ]:
def analyze_hu_windowing(volume_index, vids, n_samples=500):
    all_pairs = [(vid, sid) for vid in vids if vid in volume_index['image_paths']
                 for sid in range(len(volume_index['image_paths'][vid]))]
    sampled = random.sample(all_pairs, min(n_samples, len(all_pairs)))
    intensities = []
    for vid, sid in sampled:
        img = np.array(Image.open(volume_index['image_paths'][vid][sid]).convert('L'),
                       dtype=np.float32) / 255.0
        intensities.extend(img.flatten())
    intensities = np.array(intensities)
    percentiles = {
        'p5': float(np.percentile(intensities, 5)),
        'p25': float(np.percentile(intensities, 25)),
        'p50': float(np.percentile(intensities, 50)),
        'p75': float(np.percentile(intensities, 75)),
        'p95': float(np.percentile(intensities, 95)),
    }
    hu_windows = {
        'percentiles': percentiles,
        'recommended': {'name': 'liver_soft_tissue', 'low': -100, 'high': 400},
        'alternatives': [
            {'name': 'abdomen', 'low': -100, 'high': 400},
            {'name': 'liver_soft', 'low': -30, 'high': 150},
            {'name': 'liver_bone', 'low': 30, 'high': 300},
        ],
        'visualization_needed': True,
    }
    return intensities, percentiles, hu_windows

print("[WINDOW] Computing HU windowing analysis...")
intensities, percentiles, hu_windows = analyze_hu_windowing(volume_index, train_vids)

print(f"   Percentiles:")
for k, v in percentiles.items():
    print(f"     {k}: {v:.4f}")

# Histogram with percentiles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(intensities, bins=80, color='steelblue', edgecolor='black', alpha=0.7)
for pname, pval in percentiles.items():
    axes[0].axvline(pval, linestyle='--', alpha=0.7, label=f"{pname}={pval:.3f}")
axes[0].set_xlabel('Normalized Intensity (0-1)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Intensity Distribution with Percentiles')
axes[0].legend(fontsize=8)

# HU window comparison
window_names = [w['name'] for w in hu_windows['alternatives']]
window_ranges = [f"[{w['low']}, {w['high']}]" for w in hu_windows['alternatives']]
window_midpoints = [(w['low'] + w['high']) / 2 for w in hu_windows['alternatives']]
y_pos = range(len(window_names))
axes[1].barh(y_pos, window_midpoints, color='lightcoral', edgecolor='black')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(window_names)
for i, (name, rng) in enumerate(zip(window_names, window_ranges)):
    axes[1].text(window_midpoints[i], i, f'  {rng}', va='center', fontsize=9)
axes[1].set_xlabel('Window Center')
axes[1].set_title('HU Window Alternatives')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(EDA_PLOTS_DIR / '09_hu_windowing.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {EDA_PLOTS_DIR / '09_hu_windowing.png'}")

## [4B] CLAHE Demo
**Purpose:** Preview CLAHE contrast enhancement with tumor overlay and histogram comparison


In [ ]:
import cv2

def apply_clahe(img, clip=2.0, grid=(8, 8)):
    img_u8 = (img * 255).astype(np.uint8)
    result = cv2.createCLAHE(clipLimit=clip, tileGridSize=grid).apply(img_u8)
    return result.astype(np.float32) / 255.0

def show_clahe_with_histograms(volume_index, vids, n=4):
    all_pairs = [(vid, sid) for vid in vids if vid in volume_index['image_paths']
                 for sid in range(len(volume_index['image_paths'][vid]))]
    sampled = random.sample(all_pairs, min(n, len(all_pairs)))
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    for i, (vid, sid) in enumerate(sampled):
        img = np.array(Image.open(volume_index['image_paths'][vid][sid]).convert('L'),
                       dtype=np.float32) / 255.0
        enhanced = apply_clahe(img)
        mask = None
        if vid in volume_index['mask_paths'] and sid < len(volume_index['mask_paths'][vid]):
            mask = np.array(Image.open(volume_index['mask_paths'][vid][sid]).convert('L'),
                           dtype=np.float32) / 255.0
        axes[i, 0].imshow(img, cmap='gray')
        if mask is not None and mask.sum() > 0:
            axes[i, 0].imshow(np.ma.masked_where(mask == 0, mask), cmap='Reds', alpha=0.4)
        axes[i, 0].set_title(f'Original Vol{vid} Slice{sid}')
        axes[i, 0].axis('off')
        axes[i, 1].imshow(enhanced, cmap='gray')
        if mask is not None and mask.sum() > 0:
            axes[i, 1].imshow(np.ma.masked_where(mask == 0, mask), cmap='Reds', alpha=0.4)
        axes[i, 1].set_title('CLAHE Enhanced')
        axes[i, 1].axis('off')
        axes[i, 2].hist(img.flatten(), bins=50, color='steelblue', alpha=0.7)
        axes[i, 2].set_xlim(0, 1)
        axes[i, 2].set_title('Original Histogram')
        axes[i, 3].hist(enhanced.flatten(), bins=50, color='darkorange', alpha=0.7)
        axes[i, 3].set_xlim(0, 1)
        axes[i, 3].set_title('CLAHE Histogram')
    plt.suptitle('CLAHE Contrast Enhancement (clip=2.0, tile=8x8)', fontsize=14)
    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR / '10_clahe_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[OK] Saved: {EDA_PLOTS_DIR / '10_clahe_comparison.png'}")

print("[CLAHE] Computing CLAHE enhancement preview...")
show_clahe_with_histograms(volume_index, train_vids)

## [5B] 2.5D Context Analysis
**Purpose:** Analyze tumor continuity to determine optimal context window


In [ ]:
def analyze_tumor_thickness(volume_index, vids, n_vols=20):
    sampled_vids = random.sample([v for v in vids if v in volume_index['mask_paths']],
                                 min(n_vols, len(vids)))
    thicknesses = []
    for vid in sampled_vids:
        mask_paths = volume_index['mask_paths'].get(vid, [])
        if not mask_paths:
            continue
        batch = []
        for sid in range(len(mask_paths)):
            m = np.array(Image.open(mask_paths[sid]).convert('L'), dtype=np.float32) / 255.0
            batch.append((m > 0).astype(np.float32))
        if batch:
            tensor = batch_to_tensor(batch)
            has_tumor = to_numpy((tensor.sum(dim=(1, 2)) > 0))
            del tensor
            gpu_clear()
        else:
            continue
        max_run = curr = 0
        for t in has_tumor:
            if t:
                curr += 1
                max_run = max(max_run, curr)
            else:
                curr = 0
        if max_run > 0:
            thicknesses.append(max_run)
    return thicknesses

print("[CONTEXT] Analyzing tumor thickness across volumes...")
thicknesses = analyze_tumor_thickness(volume_index, train_vids)

if thicknesses:
    thickness_arr = np.array(thicknesses)
    mean_th = float(thickness_arr.mean())
    max_th = int(thickness_arr.max())
    p25_th = int(np.percentile(thickness_arr, 25))
    p75_th = int(np.percentile(thickness_arr, 75))
    median_th = int(np.median(thickness_arr))
    recommended = max(1, int(np.ceil(p25_th / 3)))
    print(f"   Volumes analyzed: {len(thicknesses)}")
    print(f"   Mean thickness: {mean_th:.1f} slices")
    print(f"   Median thickness: {median_th} slices")
    print(f"   Max thickness: {max_th} slices")
    print(f"   P25 thickness: {p25_th}, P75 thickness: {p75_th}")
    print(f"   [INFO] Recommended context window: {recommended} slices (based on P25/3)")

    # Histogram
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(thicknesses, bins=min(30, len(set(thicknesses))), color='teal',
            edgecolor='black', alpha=0.7)
    ax.axvline(mean_th, color='red', linestyle='--', label=f'Mean: {mean_th:.1f}')
    ax.axvline(median_th, color='green', linestyle='--', label=f'Median: {median_th}')
    ax.axvline(p25_th, color='orange', linestyle=':', label=f'P25: {p25_th}')
    ax.axvline(p75_th, color='purple', linestyle=':', label=f'P75: {p75_th}')
    ax.set_xlabel('Tumor Thickness (consecutive slices)')
    ax.set_ylabel('Number of Volumes')
    ax.set_title('Distribution of Tumor Thickness Across Volumes')
    ax.legend()
    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR / '11_tumor_thickness.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[OK] Saved: {EDA_PLOTS_DIR / '11_tumor_thickness.png'}")

    hist_counts, hist_edges = np.histogram(thicknesses, bins=min(20, len(set(thicknesses))))
    context_stats = {
        'sample_volumes': len(thicknesses),
        'mean_thickness': round(mean_th, 2),
        'median_thickness': median_th,
        'max_thickness': max_th,
        'p25_thickness': p25_th,
        'p75_thickness': p75_th,
        'recommended_context': recommended,
        'distribution': {
            'counts': [int(c) for c in hist_counts],
            'edges': [round(float(e), 1) for e in hist_edges],
        },
    }
else:
    context_stats = {'sample_volumes': 0, 'mean_thickness': 0, 'max_thickness': 0, 'recommended_context': 3}
    print("   [WARNING] No tumor data found for analysis")

## [6B] Augmentation Preview
**Purpose:** Verify tumor-mask alignment across geometric and intensity augmentations


In [ ]:
def preview_augmentations(volume_index, vids, n=3):
    all_pairs = [(vid, sid) for vid in vids if vid in volume_index['mask_paths']
                 for sid in range(len(volume_index['mask_paths'][vid]))]
    sampled = random.sample(all_pairs, min(n, len(all_pairs)))
    augs = [
        ('Original', lambda x: x, lambda x: x),
        ('H-Flip', np.fliplr, np.fliplr),
        ('V-Flip', np.flipud, np.flipud),
        ('Rot90', lambda x: np.rot90(x, k=1), lambda x: np.rot90(x, k=1)),
        ('Rot180', lambda x: np.rot90(x, k=2), lambda x: np.rot90(x, k=2)),
        ('Rot270', lambda x: np.rot90(x, k=3), lambda x: np.rot90(x, k=3)),
        ('Bright+', lambda x: np.clip(x * 1.3, 0, 1), lambda x: x),
        ('Contrast', lambda x: np.clip((x - 0.5) * 1.5 + 0.5, 0, 1), lambda x: x),
    ]
    fig, axes = plt.subplots(len(sampled), len(augs), figsize=(4 * len(augs), 4 * n))
    for r, (vid, sid) in enumerate(sampled):
        img = np.array(Image.open(volume_index['image_paths'][vid][sid]).convert('L'),
                       dtype=np.float32) / 255.0
        mask = np.array(Image.open(volume_index['mask_paths'][vid][sid]).convert('L'),
                       dtype=np.float32) / 255.0
        for c, (name, img_tf, msk_tf) in enumerate(augs):
            ai = img_tf(img.copy())
            am = msk_tf(mask.copy())
            axes[r, c].imshow(ai, cmap='gray')
            if am.sum() > 0:
                axes[r, c].imshow(np.ma.masked_where(am == 0, am), cmap='Reds', alpha=0.4)
            axes[r, c].set_title(name if r == 0 else '', fontsize=9)
            axes[r, c].axis('off')
    plt.suptitle('Augmentation Preview - Tumor-Mask Alignment Verified', fontsize=13)
    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR / '12_augmentation_preview.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[OK] Saved: {EDA_PLOTS_DIR / '12_augmentation_preview.png'}")

print("[AUG] Previewing augmentations with tumor-mask alignment...")
preview_augmentations(volume_index, train_vids)

## [7B] Outlier Identification
**Purpose:** Flag unusual volumes for special handling and generate outlier summary


In [ ]:
def identify_outliers(volume_df):
    thresholds = {
        'min_slices': 100,
        'max_slices': 800,
    }
    report = {
        'thresholds': thresholds,
        'no_tumor_volumes': [],
        'high_tumor_volumes': [],
        'few_slices_volumes': [],
        'many_slices_volumes': [],
        'large_tumor_volumes': [],
    }
    for _, row in volume_df.iterrows():
        vid = int(row['volume_id'])
        if row['total_slices'] < thresholds['min_slices']:
            report['few_slices_volumes'].append(vid)
        if row['total_slices'] > thresholds['max_slices']:
            report['many_slices_volumes'].append(vid)
    all_outliers = set()
    for key in ['few_slices_volumes', 'many_slices_volumes']:
        all_outliers.update(report[key])
    report['total_outliers'] = len(all_outliers)
    print(f"[OUTLIER] Outlier Report:")
    print(f"   Thresholds: {thresholds}")
    for key in ['few_slices_volumes', 'many_slices_volumes']:
        print(f"   {key}: {len(report[key])} volumes")
    print(f"   Total unique outliers: {report['total_outliers']}")
    return report, all_outliers

print("[OUTLIER] Identifying outlier volumes...")
outlier_report, outlier_ids = identify_outliers(volume_df)

# Scatter plot with outliers highlighted
fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {'train': '#2ecc71', 'val': '#3498db', 'test': '#e74c3c'}
for name in ['train', 'val', 'test']:
    sub = volume_df[volume_df['split'] == name]
    ax.scatter(sub['volume_id'], sub['total_slices'],
               c=colors_map[name], label=name, alpha=0.6, s=60)
outlier_df = volume_df[volume_df['volume_id'].isin(outlier_ids)]
ax.scatter(outlier_df['volume_id'], outlier_df['total_slices'],
           facecolors='none', edgecolors='red', s=120, linewidths=2, label='Outlier')
ax.set_xlabel('Volume ID')
ax.set_ylabel('Total Slices')
ax.set_title('Volume Distribution with Outliers Highlighted')
ax.legend()
plt.tight_layout()
plt.savefig(EDA_PLOTS_DIR / '13_outlier_volumes.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {EDA_PLOTS_DIR / '13_outlier_volumes.png'}")

# Save outlier list to JSON
outlier_json_path = EDA_OUTPUT_DIR / 'outlier_volumes.json'
with open(outlier_json_path, 'w') as f:
    json.dump({
        'outlier_volume_ids': sorted(int(v) for v in outlier_ids),
        'report': outlier_report,
    }, f, indent=2)
print(f"[OK] Saved: {outlier_json_path}")

# Save enhanced consolidated JSON
enhanced_data = {
    'enhanced_stats': enhanced_stats,
    'hu_windows': hu_windows,
    'context_stats': context_stats,
    'outlier_report': outlier_report,
}
enhanced_path = EDA_OUTPUT_DIR / 'enhanced_volume_stats.json'
with open(enhanced_path, 'w') as f:
    json.dump(enhanced_data, f, indent=2)
print(f"[OK] Saved: {enhanced_path}")

## [SAVE] Cell 9: Save Statistics JSON

In [ ]:
# Aggregate all statistics into single dictionary
phase2_data = {
    'phase': 'Phase 2 - Exploratory Data Analysis',
    'date': '2026-05-21',
    'image_size': (256, 256),
    'dataset_summary': {
        'total_volumes': len(volume_index['volumes']),
        'total_slices': total_slices,
        'train_volumes': len(train_vids),
        'val_volumes': len(val_vids),
        'test_volumes': len(test_vids),
    },
    'intensity_statistics': intensity_stats,
    'tumor_statistics': tumor_stats,
    'class_imbalance': imbalance_by_split,
    'morphology_statistics': morpho_stats,
    'volume_statistics': {
        split: {
            'mean_slices_per_volume': float(np.mean(vol_stats[split]['slices_per_volume'])),
            'mean_tumor_pixels': float(np.mean(vol_stats[split]['tumor_pixels'])) if vol_stats[split]['tumor_pixels'] else 0,
        } for split in ['train', 'val', 'test']
    },
}

# Save machine-readable version (for Phase 3)
metadata_path = DatasetConfig.METADATA_DIR / 'phase2_statistics.json'
DatasetConfig.METADATA_DIR.mkdir(parents=True, exist_ok=True)
with open(metadata_path, 'w') as f:
    json.dump(phase2_data, f, indent=2)
print(f"[OK] Machine statistics -> {metadata_path}")

# Save human-readable EDA report
eda_report_path = EDA_OUTPUT_DIR / 'statistics.json'
with open(eda_report_path, 'w') as f:
    json.dump(phase2_data, f, indent=2)
print(f"[OK] EDA report -> {eda_report_path}")

## [INSIGHTS] Cell 10: Key Insights & Recommendations

In [ ]:
tr = tumor_stats
imb = imbalance_by_split['train']
spv = [vol_stats[s]['slices_per_volume'] for s in ['train', 'val', 'test']]
min_s, max_s = min(min(v) for v in spv), max(max(v) for v in spv)

print("=" * 60)
print("  PHASE 2 EDA - KEY INSIGHTS")
print("=" * 60)
print("\n[DATASET OVERVIEW]")
print(f"  Volumes: {len(volume_index['volumes'])}")
print(f"  Total slices: {total_slices:,}")
print(f"  Image size: 256x256 (resized from 512x512)")
print(f"  Split: Train={len(train_vids)}v / Val={len(val_vids)}v / Test={len(test_vids)}v")
print()
print('[FINDING 1: EXTREME CLASS IMBALANCE]')
print(f"  Tumor occupies only {imb['tumor_pct']:.4f}% of pixels")
print(f"  Background-to-tumor ratio: 1:{imb['imbalance_ratio']:.0f}")
print(f"  {100 - tr['tumor_slice_pct']:.1f}% of slices have NO tumor")
print("  -> Standard accuracy metric will be misleading")
print("  -> Use Dice coefficient as PRIMARY metric")
print("  -> Use BCE + Dice combined loss function")
print()
print('[FINDING 2: TUMOR SIZE VARIABILITY]')
print(f"  Mean tumor coverage: {tr['mean_coverage_pct']:.3f}%")
print(f"  Max tumor coverage: {tr['max_coverage_pct']:.3f}%")
print(f"  Mean region size: {morpho_stats['mean_region_size']:.0f} pixels")
print(f"  Mean aspect ratio: {morpho_stats['mean_aspect_ratio']:.2f} (slightly elongated)")
print("  -> Model must handle multi-scale tumors")
print("  -> Data augmentation (rotation, scaling) is crucial")
print()
print('[FINDING 3: VOLUME VARIATION]')
print(f"  Slices per volume range: {min_s} to {max_s}")
print("  -> Consistent preprocessing pipeline is essential")
print("  -> Model must be robust to varying input depths")
print()
print('[FINDING 4: SPLIT QUALITY]')
print("  All 3 splits show similar class imbalance ratios")
print("  -> No systematic bias detected in val/test sets")

print("=" * 60)
print("  RECOMMENDATIONS")
print("=" * 60)
print()
print('1. LOSS FUNCTION: BCE + Dice combined loss')
print('   - BCE: pixel-wise accuracy')
print('   - Dice: handles extreme class imbalance')
print()
print('2. PRIMARY METRIC: Dice coefficient')
print('   - More clinically relevant than accuracy')
print('   - Range [0,1], higher is better')
print()
print('3. AUGMENTATION: Geometric transforms')
print('   - Flips, rotations, elastic deformations')
print('   - Intensity shifts for robustness')
print()
print('4. NEXT STEPS')
print('   - Phase 3: Preprocessing pipeline (HU windowing, CLAHE)')
print('   - Phase 4: Model Development (U-Net / MobileNetV2)')
print('   - Phase 5: Training with BCE + Dice loss')

print("=" * 60)
print("  PHASE 2 EDA COMPLETE")
print("=" * 60)
print(f"\nPlots saved (7 files):")
print(f"  {EDA_PLOTS_DIR}/")
print(f"\nStatistics JSON:")
print(f"  metadata: {metadata_path}")
print(f"  EDA:      {eda_report_path}")
print("=" * 60)